# Adaptive Traffic Light — ML Notebook

Trains a small neural network to decide **which road gets the green light** and **for how long**, based on current queue levels and traffic intensity.

The model learns in a simulation, then is exported to **TFLite** for deployment on embedded hardware.

---

## 1. Imports & Configuration

Global constants that control the simulation and model behaviour. Adjust `MIN_GREEN` and `MAX_GREEN` to change the range of allowed green-phase durations.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

FIGURES_DIR = "../diagrams/graphs"
os.makedirs(FIGURES_DIR, exist_ok=True)

# ── Configurable constants ────────────────────────────────────────────────────
STEP_S    = 0.5    # seconds per simulation tick (500 ms)
MIN_GREEN = 5.0    # minimum green phase (seconds)
MAX_GREEN = 15.0   # maximum green phase (seconds)

# Queue dynamics per 500 ms tick.
# Halved from per-second values so real-time drain/accum is unchanged:
#   e.g. DRAIN_RATE 0.06/s × 10 s  =  0.03/tick × 20 ticks  =  0.60
DRAIN_RATE = 0.03    # green road loses this fraction per tick
ACCUM_BASE = 0.015   # red road base accumulation per tick (× intensity multiplier)

INTENSITY_MAP = {"low": 0.3, "medium": 0.6, "high": 1.0}

np.random.seed(42)
tf.random.set_seed(42)

## 2. Traffic Simulation

`TrafficSim` models a 4-way intersection as four normalised queues (0 = empty, 1 = full).

**Time granularity:** One simulation tick = 500 ms (`STEP_S`). Duration is always specified in whole ticks internally, but exposed as seconds everywhere else in the notebook.

**Queue dynamics per phase:**
- The **green road** drains: `queue -= DRAIN_RATE × ticks`. At the default rate, a fully-loaded road takes ~33 ticks (≈ 17 s) to reach zero.
- All **red roads** accumulate: `queue += ACCUM_BASE × intensity × ticks`. A medium-intensity road gains ~0.009 per tick; a high-intensity road gains ~0.015.
- Queues are clamped to [0, 1] — they can't go negative or overflow.

**System balance:** With 1 green road out of 4 in round-robin, drain rate (0.03) exceeds total accumulation on the green road (0 from it) minus accumulation on 3 red roads (3 × 0.009 = 0.027 per tick). Net drain = 0.003 per tick — the system slowly clears under round-robin, which is why round-robin sets the performance baseline.

`observe()` returns the 8-element input vector the model will receive: `[q₀, q₁, q₂, q₃, i₀, i₁, i₂, i₃]`.

In [6]:
class TrafficSim:
    """4-way intersection simulator.

    queues      -- normalised occupancy per road, clamped [0, 1]
    intensities -- arrival-rate multipliers (from INTENSITY_MAP)
    """

    def __init__(self, intensities, init_queues=None):
        self.intensities = np.array(intensities, dtype=np.float32)
        if init_queues is None:
            self.queues = np.random.uniform(0.1, 0.9, size=4).astype(np.float32)
        else:
            self.queues = np.array(init_queues, dtype=np.float32)

    def step(self, green_road: int, duration_ticks: int):
        """Advance simulation by duration_ticks × STEP_S seconds."""
        for road in range(4):
            if road == green_road:
                self.queues[road] -= DRAIN_RATE * duration_ticks
            else:
                self.queues[road] += ACCUM_BASE * self.intensities[road] * duration_ticks
        self.queues = np.clip(self.queues, 0.0, 1.0)

    def observe(self) -> np.ndarray:
        """Return [q_top, q_bottom, q_left, q_right, i_top, i_bottom, i_left, i_right]."""
        return np.concatenate([self.queues, self.intensities]).astype(np.float32)

    def reset(self, init_queues=None):
        if init_queues is None:
            self.queues = np.random.uniform(0.1, 0.9, size=4).astype(np.float32)
        else:
            self.queues = np.array(init_queues, dtype=np.float32)

## 3. Greedy Policy (Label Generation)

The **teacher** the supervised model imitates. At each decision point it:
1. Scores each road as `queue × intensity` — heavier, busier roads score higher.
2. Picks the road with the highest score.
3. Sets the duration proportional to that road's queue, scaled to `[MIN_GREEN, MAX_GREEN]`.

**Why use a greedy policy as the teacher?**  
A greedy heuristic is simple to define, always produces a deterministic answer, and performs well on unbalanced traffic where one road clearly needs priority. It gives us a clean, noise-free set of training labels. The neural network's job is to learn to replicate this policy from the raw observation vector — then run it in hardware without needing the Python logic.

**Limitation:** The greedy policy isn't globally optimal for balanced traffic (where round-robin is better), and it doesn't plan ahead — it only looks at the current state.

In [7]:
def greedy_decision(queues: np.ndarray, intensities: np.ndarray):
    """Pick road with highest weighted queue; return (road, duration_seconds)."""
    scores = queues * intensities
    road = int(np.argmax(scores))
    duration_s = float(np.clip(
        MIN_GREEN + queues[road] * (MAX_GREEN - MIN_GREEN),
        MIN_GREEN, MAX_GREEN,
    ))
    return road, duration_s

## 4. Dataset Generation

Runs the simulation for `n_steps` phases under the greedy policy and records:

| Array | Shape | Meaning |
|---|---|---|
| `X` | (n_steps, 8) | Observation at each decision point |
| `y_road` | (n_steps, 4) | One-hot road chosen by greedy policy |
| `y_dur` | (n_steps, 1) | Duration in **seconds** `[MIN_GREEN, MAX_GREEN]` |

**Multi-config training:** The dataset cycles through varied intensity combinations every `steps_per_config` phases. This ensures the model sees all four intensity inputs vary during training — if all configs were all-medium, the intensity features `[i₀, i₁, i₂, i₃]` would be constant and the model would learn to ignore them, causing it to perform worse than round-robin on imbalanced traffic.

In [ ]:
def generate_dataset(n_steps=2000, steps_per_config=50):
    """Run greedy simulation across varied intensity configs and collect (observation, label) pairs.

    Cycles through a fixed set of intensity combinations so the model sees all
    four intensity inputs vary during training — not just all-medium.
    """
    configs = [
        [INTENSITY_MAP["medium"]] * 4,
        [INTENSITY_MAP["high"],   INTENSITY_MAP["low"],    INTENSITY_MAP["low"],    INTENSITY_MAP["low"]],
        [INTENSITY_MAP["low"],    INTENSITY_MAP["high"],   INTENSITY_MAP["low"],    INTENSITY_MAP["low"]],
        [INTENSITY_MAP["low"],    INTENSITY_MAP["low"],    INTENSITY_MAP["high"],   INTENSITY_MAP["low"]],
        [INTENSITY_MAP["low"],    INTENSITY_MAP["low"],    INTENSITY_MAP["low"],    INTENSITY_MAP["high"]],
        [INTENSITY_MAP["high"],   INTENSITY_MAP["high"],   INTENSITY_MAP["low"],    INTENSITY_MAP["low"]],
        [INTENSITY_MAP["high"],   INTENSITY_MAP["low"],    INTENSITY_MAP["high"],   INTENSITY_MAP["low"]],
        [INTENSITY_MAP["low"],    INTENSITY_MAP["high"],   INTENSITY_MAP["low"],    INTENSITY_MAP["high"]],
        [INTENSITY_MAP["high"],   INTENSITY_MAP["medium"], INTENSITY_MAP["low"],    INTENSITY_MAP["medium"]],
        [INTENSITY_MAP["low"],    INTENSITY_MAP["medium"], INTENSITY_MAP["high"],   INTENSITY_MAP["medium"]],
    ]

    X, y_road, y_dur = [], [], []
    config_idx = 0
    steps_in_config = 0

    sim = TrafficSim(configs[0])

    for _ in range(n_steps):
        if steps_in_config >= steps_per_config:
            config_idx = (config_idx + 1) % len(configs)
            sim.intensities = np.array(configs[config_idx], dtype=np.float32)
            steps_in_config = 0

        obs = sim.observe()
        road, duration_s = greedy_decision(sim.queues, sim.intensities)

        one_hot = np.zeros(4, dtype=np.float32)
        one_hot[road] = 1.0

        X.append(obs)
        y_road.append(one_hot)
        y_dur.append([duration_s])

        sim.step(road, round(duration_s / STEP_S))
        steps_in_config += 1

    return (
        np.array(X),
        np.array(y_road),
        np.array(y_dur, dtype=np.float32),
    )


X, y_road, y_dur = generate_dataset(n_steps=2000, steps_per_config=50)
print(f"Dataset: X={X.shape}, y_road={y_road.shape}, y_dur={y_dur.shape}")
print(f"Road label distribution: {y_road.sum(axis=0).astype(int)}")
print(f"Duration range: {y_dur.min():.1f}s - {y_dur.max():.1f}s")
print(f"Intensity range seen: {X[:, 4:].min():.2f} - {X[:, 4:].max():.2f}")

### Dataset Observations

**Road distribution** — with varied configs the distribution is no longer `[500 500 500 500]`. Configs with one dominant high-intensity road cause the greedy policy to pick that road more often. This is expected: the model learns to prioritise busy roads.

**Intensity range `0.30 – 1.00`** — confirms the model sees the full `[low, medium, high]` intensity range and will learn to use those features. If this shows `0.60 – 0.60`, the multi-config cycling is not working and the model will ignore intensity inputs.

## 5. Supervised Model Architecture

A small two-headed network with a shared backbone:

```
Input (8)
  └─ Dense(16, ReLU)
       └─ Dense(8, ReLU)
            ├─ road     → Dense(4, softmax)  # which road gets green
            └─ duration → Dense(1, linear)   # green duration in seconds
```

**Loss:** categorical cross-entropy on road + MSE on duration (raw seconds), weighted equally.  
At inference: `road = argmax(softmax)`, `duration = clamp(linear_out, MIN_GREEN, MAX_GREEN)`.

In [9]:
def build_model():
    inp = keras.Input(shape=(8,), name="observation")
    x = keras.layers.Dense(16, activation="relu")(inp)
    x = keras.layers.Dense(8,  activation="relu")(x)
    road_out = keras.layers.Dense(4, activation="softmax", name="road")(x)
    dur_out  = keras.layers.Dense(1, activation="linear",  name="duration")(x)
    model = keras.Model(inputs=inp, outputs=[road_out, dur_out])
    model.compile(
        optimizer="adam",
        loss={"road": "categorical_crossentropy", "duration": "mse"},
        # Duration MSE on raw seconds (~100 scale) vs road CE (~1.4 scale).
        # Weight 0.01 keeps both losses comparable so the road head is not starved.
        loss_weights={"road": 1.0, "duration": 0.01},
        metrics={"road": "accuracy"},
    )
    return model


model = build_model()
model.summary()

I0000 00:00:1778833867.347559      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ observation         │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 16)        │        144 │ observation[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 8)         │        136 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ road (Dense)        │ (None, 4)         │         36 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ duration (Dense)    │ (None, 1)         │          9 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 325 (1.27 KB)

 Trainable params: 325 (1.27 KB)

 Non-trainable params: 0 (0.00 B)

## 6. Training

50 epochs with a 90/10 train/validation split.

**What to look for in the output:**
- `val_road_accuracy → 1.000` means the model correctly picks the same road as the greedy policy on every validation sample. Since the greedy rule is deterministic (argmax of queue × intensity), the network only needs to learn a simple linear ranking — it typically nails this within 10–20 epochs.
- `val_duration_MSE → ~0.0` means the model's predicted duration is within a fraction of a second of the greedy target. The linear activation lets the network output any value; Adam finds the right weights quickly.
- If `val_road_accuracy` stalls below 0.95, the duration loss weight (`loss_weights`) is too high and is dominating the gradient — the road head doesn't get enough signal. The current weight of 0.01 on duration keeps both losses comparable.

In [10]:
history = model.fit(
    X,
    {"road": y_road, "duration": y_dur},
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    verbose=1,
)

val_road_acc = history.history["val_road_accuracy"][-1]
val_dur_loss = history.history["val_duration_loss"][-1]
print(f"\nFinal val road accuracy: {val_road_acc:.3f}")
print(f"Final val duration MSE:  {val_dur_loss:.4f}")

Epoch 1/50


I0000 00:00:1778833869.970590     124 service.cc:152] XLA service 0x7a3a1400ab70 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778833869.970622     124 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1778833870.275334     124 cuda_dnn.cc:529] Loaded cuDNN version 91002


55/57 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - duration_loss: 119.2785 - loss: 2.5479 - road_accuracy: 0.3587 - road_loss: 1.3551

I0000 00:00:1778833871.065084     124 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


57/57 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - duration_loss: 119.2151 - loss: 2.5467 - road_accuracy: 0.3627 - road_loss: 1.3545 - val_duration_loss: 115.4110 - val_loss: 2.4709 - val_road_accuracy: 0.5000 - val_road_loss: 1.3168
Epoch 2/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - duration_loss: 114.7445 - loss: 2.4472 - road_accuracy: 0.5122 - road_loss: 1.2998 - val_duration_loss: 110.3716 - val_loss: 2.3509 - val_road_accuracy: 0.5000 - val_road_loss: 1.2471
Epoch 3/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - duration_loss: 109.1405 - loss: 2.3111 - road_accuracy: 0.6100 - road_loss: 1.2196 - val_duration_loss: 102.7444 - val_loss: 2.1693 - val_road_accuracy: 0.7500 - val_road_loss: 1.1418
Epoch 4/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - duration_loss: 100.5513 - loss: 2.1116 - road_accuracy: 0.7578 - road_loss: 1.1059 - val_duration_loss: 90.6582 - val_loss: 1.9109 - val_road_accuracy: 0.7500 - val_road_loss: 1.0043
Epoch 5/50
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - duration_loss: 

### Training Results

- **`val_road_accuracy = 1.000`** — the model picks the correct road on every validation example. Road selection is essentially a "which of these 4 numbers is largest?" problem after the greedy labels are computed; the network learns this ranking within the first ~15 epochs.
- **`val_duration_MSE ≈ 0.0`** — the model predicts green duration to within a fraction of a second. The linear output head has no activation constraint, so it directly fits the [10.9, 15.0] second range without any scaling needed.

These results confirm the supervised model has successfully learned to replicate the greedy policy. At inference it will make the same road choices as the handcrafted rule, and output durations close to what the formula would give — all from the 8-float observation vector, without running any Python logic on the device.

## 7. Export to TFLite — Supervised

Converts the trained supervised model to a **TFLite flatbuffer** (`traffic_model.tflite`) with no quantization, preserving full float32 precision.

> **Note:** `TFLiteConverter.from_keras_model` and `model.export()` both have a bug with multi-output models in TF 2.20. The workaround is to wrap the model in a `tf.function` with an explicit input signature and convert from the resulting concrete function.

In [11]:
# TFLiteConverter.from_keras_model and model.export() both break on multi-output
# models in TF 2.20. Converting via a concrete function works correctly.
run_fn = tf.function(
    lambda x: model(x),
    input_signature=[tf.TensorSpec(shape=[None, 8], dtype=tf.float32)],
)
converter = tf.lite.TFLiteConverter.from_concrete_functions(
    [run_fn.get_concrete_function()], trackable_obj=model
)
tflite_model = converter.convert()

tflite_path = "traffic_model.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print(f"Exported {len(tflite_model):,} bytes -> {tflite_path}")

Exported 3,520 bytes -> traffic_model.tflite


I0000 00:00:1778833884.808956      57 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
I0000 00:00:1778833884.809140      57 single_machine.cc:374] Starting new session
I0000 00:00:1778833884.810072      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
W0000 00:00:1778833884.849568      57 tf_tfl_flatbuffer_helpers.cc:365] Ignored output_format.
W0000 00:00:1778833884.849598      57 tf_tfl_flatbuffer_helpers.cc:368] Ignored drop_control_dependency.


In [ ]:
# ── Export weights as C++ header ───────────────────────────────────────────
# The ESP32 firmware uses compiled-in float arrays instead of loading a TFLite
# file at runtime. This avoids the TFLite interpreter overhead on a constrained
# microcontroller and lets the weights live in flash with zero heap allocation.

def arr_to_c(arr, name):
    flat = arr.flatten(order='C')
    vals = ', '.join(f'{v:.8f}f' for v in flat)
    return f'const float {name}[] = {{{vals}}};'

lines = [
    '#pragma once',
    '// Auto-generated by traffic_light.ipynb — do not edit manually',
    '// Supervised model: 8->16->8->[4,1]',
]

sup_layers = [l for l in model.layers if l.get_weights()]
assert len(sup_layers) == 4, f'Expected 4 layers, got {len(sup_layers)}: {[l.name for l in sup_layers]}'
for layer, cname in zip(sup_layers, ['SUP_L1', 'SUP_L2', 'SUP_ROAD', 'SUP_DUR']):
    W, b = layer.get_weights()
    lines.append(arr_to_c(W, f'{cname}_W'))
    lines.append(arr_to_c(b, f'{cname}_B'))

out_path = '../src/ml_weights.h'
with open(out_path, 'w', newline='\n') as f:
    f.write('\n'.join(lines) + '\n')

sup_shapes = [(l.name, l.get_weights()[0].shape) for l in sup_layers]
print(f'Written -> {out_path}')
print('Supervised:', sup_shapes)

## 8. Benchmark

Compares two control strategies over a fixed 500-step episode:

| Mode | Road selection | Duration |
|---|---|---|
| **Normal** | Round-robin (top → bottom → left → right) | Fixed = `min_green` seconds |
| **Supervised** | `argmax` of model softmax output | Linear output clamped to `[min_green, MAX_GREEN]` |

The metric is the **average total queue across all roads** per phase (lower = less congestion).

In [ ]:
def run_episode(mode: str, intensities, min_green: float, model=None, steps: int = 500):
    """Simulate one episode; return per-phase total queue array.

    mode: 'normal'     -- fixed round-robin, duration = min_green seconds
          'supervised' -- supervised model (greedy imitation)
    """
    sim = TrafficSim(intensities)
    total_waits = []
    rr_idx = 0

    for _ in range(steps):
        obs = sim.observe()

        if mode == "normal":
            road = rr_idx % 4
            duration_s = float(min_green)
            rr_idx += 1
        else:
            road_probs, dur_out = model(obs[np.newaxis], training=False)
            road = int(np.argmax(road_probs[0]))
            duration_s = float(np.clip(dur_out[0, 0], min_green, MAX_GREEN))

        sim.step(road, round(duration_s / STEP_S))
        total_waits.append(float(sim.queues.sum()))

    return np.array(total_waits)

## 9. Results Table

Sweeps `min_green` across `[5, 7, 9, 11, 13, 15]` seconds for two intensity configurations:
- **All medium** — uniform traffic across all roads.
- **Top high, rest low** — one dominant road, the rest sparse.

Each cell shows `mean ± std` of total queue over the 500-step episode.

In [ ]:
MIN_GREEN_VALUES = [5, 7, 9, 11, 13, 15]
BENCHMARK_STEPS  = 500

INTENSITY_CONFIGS = {
    "All medium":         [INTENSITY_MAP["medium"]] * 4,
    "Top high, rest low": [
        INTENSITY_MAP["high"],
        INTENSITY_MAP["low"],
        INTENSITY_MAP["low"],
        INTENSITY_MAP["low"],
    ],
}

results = {}  # (config_name, mode) -> {min_green: (mean, std)}

for cfg_name, intensities in INTENSITY_CONFIGS.items():
    for mode, mdl in [("normal", None), ("supervised", model)]:
        key = (cfg_name, mode)
        results[key] = {}
        for mg in MIN_GREEN_VALUES:
            waits = run_episode(mode, intensities, min_green=mg, model=mdl, steps=BENCHMARK_STEPS)
            results[key][mg] = (waits.mean(), waits.std())

header = f"{'Config':<22} {'min_green':>10} {'Normal':>14} {'Supervised':>16}"
print(header)
print("-" * len(header))
for cfg_name in INTENSITY_CONFIGS:
    for mg in MIN_GREEN_VALUES:
        n_mean, n_std = results[(cfg_name, "normal")][mg]
        s_mean, s_std = results[(cfg_name, "supervised")][mg]
        print(
            f"{cfg_name:<22} {mg:>9}s"
            f"   {n_mean:.3f}+-{n_std:.3f}"
            f"   {s_mean:.3f}+-{s_std:.3f}"
        )

### Score Interpretation

Each cell shows `mean ± std` of **total queue summed across all 4 roads**, averaged over every phase of a 500-phase episode. Scale is 0–4 (0 = all roads empty, 4 = all roads completely full). Lower is better.

---

**All medium — round-robin wins:**  
When all roads have identical arrival rates, the optimal policy is perfectly fair: give every road equal time. Round-robin does exactly this. The supervised model scores slightly worse (~1.20 across settings) because the greedy policy sometimes lingers on a road whose queue is only marginally higher, letting others accumulate unnecessarily.

---

**Top high, rest low — supervised wins:**  
The top road receives 3× more cars per tick than the others. Round-robin scores 1.09–1.28 because the high-traffic road is only served 25% of the time and backs up permanently. The supervised model correctly prioritises the busy road for a ~15–30% improvement over round-robin.

**Why the supervised advantage shrinks at high min_green:**  
As `min_green` grows, every phase lasts longer regardless of what the model outputs. At `min_green = 15s` (= `MAX_GREEN`), all modes are forced to use exactly 15s phases — there is nothing left to optimise. This is why the two lines converge at the right edge of the plot.

---

**Takeaway:**  
Round-robin is optimal for balanced traffic. For imbalanced traffic the greedy supervised model reliably outperforms it by prioritising high-load roads.

## 10. Plot

Visualises the benchmark results. Each subplot shows one intensity configuration:
- **Solid blue** — normal round-robin baseline.
- **Dashed red** — supervised model (greedy imitation).
- Shaded bands show ±1 standard deviation.

**How to read the plot:**
- A line that stays **low and flat** = consistently efficient scheduling.
- A line that **rises with min_green** = duration-limited — longer forced phases reduce flexibility.
- Left (balanced traffic): round-robin wins. Right (imbalanced traffic): supervised wins.

In [ ]:
fig, axes = plt.subplots(1, len(INTENSITY_CONFIGS), figsize=(12, 5), sharey=True)
fig.suptitle("Avg total queue: round-robin vs supervised", fontsize=13)

styles = [
    ("normal",     "steelblue", "-",  "Normal (round-robin)"),
    ("supervised", "tomato",    "--", "Supervised (greedy imitation)"),
]

for ax, (cfg_name, _) in zip(axes, INTENSITY_CONFIGS.items()):
    for mode, color, ls, label in styles:
        means = np.array([results[(cfg_name, mode)][mg][0] for mg in MIN_GREEN_VALUES])
        stds  = np.array([results[(cfg_name, mode)][mg][1] for mg in MIN_GREEN_VALUES])
        ax.plot(MIN_GREEN_VALUES, means, color=color, linestyle=ls, marker="o", label=label)
        ax.fill_between(MIN_GREEN_VALUES, means - stds, means + stds, color=color, alpha=0.12)

    ax.set_title(cfg_name)
    ax.set_xlabel("min_green (seconds)")
    ax.set_ylabel("Avg total queue (0–4)")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{FIGURES_DIR}/benchmark.png", dpi=120)
plt.show()
print(f"Saved {FIGURES_DIR}/benchmark.png")